# 02_silver_transform

Cleans and enriches the Bronze data:
- Type-casts all columns
- Fills missing publishers with 'Unknown'
- Adds marketing-focused derived columns (sales tier, regional %, global hit flag)
- Partitions output by `genre` for query performance

**Run 01_bronze_ingestion first.**

## Cell 1 — Read the Bronze table

`spark.table(...)` loads a saved Delta table by name. Bronze is the raw, untouched copy of the original CSV. We store it in `df_b` (dataframe bronze). `count()` tells us how many rows exist, and `display()` shows the first 5 rows so we can visually confirm the data loaded correctly.

In [0]:
# ============================================================
# CELL 1 — Read from Bronze
# ============================================================
# spark.table() reads a saved Delta table by name.
# 'gaming_bronze.vg_sales_raw' means:
#   schema (database) : gaming_bronze
#   table name        : vg_sales_raw
# We store the result in df_b so we can reference it in later cells.

df_b = spark.table('gaming_bronze.vg_sales_raw')

# f-string prints the total number of rows — a quick confirmation that data loaded.
print(f'Bronze row count: {df_b.count()}')

# display() is a Databricks built-in that renders an interactive table in the notebook.
# .limit(5) shows only the first 5 rows to avoid loading millions of records at once.
display(df_b.limit(5))


## Cell 2 — Confirm Bronze column names

Before transforming anything, we check exactly what columns exist in the Bronze table. The column names printed here are the ones we must reference in the transformation cell. If a name is misspelled or has different capitalisation, the transformation will fail.

In [0]:
# Print the list of column names from the Bronze dataframe.
# Confirm you see: 'Rank', 'Name', 'Platform', 'Year', 'Genre', 'Publisher',
#                   'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales'
# plus the three metadata columns added by Bronze: '_ingested_at', '_source_file', '_layer'
# If any expected column is missing, the Silver transformation below will fail.
print("Bronze columns:", df_b.columns)

# Show 5 rows so we can see the actual data values alongside the column names.
display(df_b.limit(5))


## Cell 3 — Count nulls in key columns

Before cleaning, we check how many missing (`null`) values exist in the columns we care about. A `null` in a sales column could mean a missing record or a data entry error. Knowing the null counts helps us decide how to handle them — drop the row, fill it, or leave it.

In [0]:
# ============================================================
# CELL 3 — Null check on key business columns
# ============================================================

# Import PySpark functions:
#   col()   — refers to a dataframe column by name
#   count() — counts non-null values in a column
#   when()  — conditional logic (like SQL CASE WHEN / Python if-else)
from pyspark.sql.functions import col, count, when

# The business columns we care about — game identity and all sales figures.
biz_cols = ['Name','Platform','Year','Genre','Publisher',
            'NA_Sales','EU_Sales','JP_Sales','Other_Sales','Global_Sales']

# For each column, count the number of null values.
# when(col(c).isNull(), c) returns the column name when the value IS null.
# count() counts those non-null results — which equals the number of nulls.
# .alias(c) renames the output column so the result table is human-readable.
null_counts = df_b.select([
    count(when(col(c).isNull(), c)).alias(c) for c in biz_cols
])

# Display results — ideally all zeros. Publisher/Year may have some nulls; that's handled later.
display(null_counts)


## Cell 4 — Create the Silver schema

In Databricks, a *schema* (also called a database) is like a folder that groups related tables. We need `gaming_silver` to exist before we can save tables into it. `CREATE SCHEMA IF NOT EXISTS` is safe to run multiple times — it creates the schema the first time and does nothing on subsequent runs.

In [0]:
# ============================================================
# CELL 4 — Create Silver schema if it does not exist
# ============================================================

# spark.sql() runs a plain SQL statement inside this Python notebook.
# 'CREATE SCHEMA IF NOT EXISTS gaming_silver' creates a new schema (database folder).
# IF NOT EXISTS means this is safe to rerun — it won't raise an error if it already exists.
spark.sql('CREATE SCHEMA IF NOT EXISTS gaming_silver')

print('Schema gaming_silver is ready.')


## Cell 5 — Build the cleaned Silver dataframe

This is the main transformation step. We take raw Bronze data and:

1. **Filter out bad rows** — remove games with no name or no sales data.
2. **Rename columns** to lowercase snake_case (e.g. `Name` → `game_name`, `Genre` → `genre`).
3. **Cast types** — convert text columns to numbers/integers where appropriate.
4. **Handle nulls** — fill missing publisher names with `'Unknown'`.
5. **Use `.select()`** to keep only the new cleaned columns.

> **Why `.select()` instead of `.drop()`?**  
> Spark is case-insensitive with column names. For example, if you use `.drop('Genre')`, Spark may also drop the new `'genre'` column you just created, because it treats them as the same name. `.select()` explicitly lists exactly the columns you want to keep, so nothing is accidentally removed.

In [0]:
# ============================================================
# CELL 5 — Build cleaned Silver dataframe
# ============================================================
# Use .select() instead of .drop() to avoid Spark's
# case-insensitive column resolution removing the new
# lowercase columns (genre, platform, publisher).

# trim() removes whitespace from the start and end of text values.
# lit() creates a fixed/hardcoded value as a column (e.g. the string 'Unknown').
# when() is conditional logic — like an if/else or SQL CASE WHEN.
from pyspark.sql.functions import trim, lit, when

# IntegerType = whole numbers (e.g. year 2006)
# DoubleType  = decimal numbers (e.g. sales figure 41.49)
from pyspark.sql.types import IntegerType, DoubleType

df_silver = (
    df_b

    # ── Step 1: Remove rows that cannot be used ───────────────────────────────
    # isNotNull() keeps only rows where the column has a real value (not null/missing).
    .filter(col('Name').isNotNull())           # Must have a game name
    .filter(col('Global_Sales').isNotNull())   # Must have a sales figure
    # Cast to Double first, then compare — avoids errors if the column is stored as text.
    .filter(col('Global_Sales').cast(DoubleType()) > 0)  # Must have sold at least something

    # ── Step 2: Rename and clean text columns ─────────────────────────────────
    # .withColumn('new_name', expression) adds a new column (or overwrites if name exists).
    # We create new lowercase names and keep the originals — they get removed in .select().
    .withColumn('game_name',  trim(col('Name')))     # 'Name'     → 'game_name'
    .withColumn('platform',   trim(col('Platform'))) # 'Platform' → 'platform'
    .withColumn('genre',      trim(col('Genre')))    # 'Genre'    → 'genre'  ← KEEP (used for partitioning)

    # Fill null publishers with 'Unknown'; otherwise trim the value.
    # when(condition, value_if_true).otherwise(value_if_false)
    .withColumn('publisher',
        when(col('Publisher').isNull(), lit('Unknown'))
        .otherwise(trim(col('Publisher')))
    )

    # ── Step 3: Cast numeric columns to proper types ──────────────────────────
    # try_cast() is like cast() but returns null instead of crashing
    # if a value cannot be converted (e.g. 'N/A' cannot become an integer).
    .withColumn('year_of_release',      col('Year').try_cast(IntegerType()))
    .withColumn('na_sales_millions',    col('NA_Sales').cast(DoubleType()))
    .withColumn('eu_sales_millions',    col('EU_Sales').cast(DoubleType()))
    .withColumn('jp_sales_millions',    col('JP_Sales').cast(DoubleType()))
    .withColumn('other_sales_millions', col('Other_Sales').cast(DoubleType()))
    .withColumn('global_sales_millions',col('Global_Sales').cast(DoubleType()))

    # ── Step 4: Keep only the new clean columns ───────────────────────────────
    # .select() picks exactly which columns survive into df_silver.
    # This is the safe alternative to .drop() — it avoids the Spark case-insensitivity bug
    # that caused 'genre' to go missing when .drop('Genre') was used.
    .select(
        'game_name', 'platform', 'genre', 'publisher',
        'year_of_release', 'na_sales_millions', 'eu_sales_millions',
        'jp_sales_millions', 'other_sales_millions', 'global_sales_millions'
    )
)

print(f'Silver row count: {df_silver.count()}')
print('Columns after cell 5:', df_silver.columns)
display(df_silver.limit(5))


## Cell 6 — Add derived marketing columns

This cell adds four new calculated columns to `df_silver`:

- **`na/eu/jp_sales_pct`** — the percentage of global sales from each region. Tells us where in the world a game sold best.
- **`is_global_hit`** — `True` if global sales ≥ 1 million copies, `False` otherwise.
- **`sales_tier`** — a category label: `Blockbuster` (10M+), `Hit` (1–10M), `Mid-Tier` (0.1–1M), `Long-Tail` (under 0.1M).

These columns make filtering and grouping much simpler in the Gold layer or dashboards.

In [0]:
# ============================================================
# CELL 6 — Add derived marketing columns
# ============================================================

# Import PySpark's round() function and rename it to avoid clashing
# with Python's built-in round() function.
from pyspark.sql.functions import round as spark_round

df_silver = (
    df_silver

    # ── Regional sales percentages ────────────────────────────────────────────
    # Formula: (region_sales / global_sales) * 100, rounded to 1 decimal.
    # Example: NA sold 41.49M out of 82.74M globally → (41.49/82.74)*100 = 50.1%
    .withColumn('na_sales_pct',
        spark_round(col('na_sales_millions') / col('global_sales_millions') * 100, 1))
    .withColumn('eu_sales_pct',
        spark_round(col('eu_sales_millions') / col('global_sales_millions') * 100, 1))
    .withColumn('jp_sales_pct',
        spark_round(col('jp_sales_millions') / col('global_sales_millions') * 100, 1))

    # ── Global hit flag ───────────────────────────────────────────────────────
    # True  = sold 1 million+ copies globally (a commercially successful title)
    # False = sold less than 1 million globally
    .withColumn('is_global_hit',
        when(col('global_sales_millions') >= 1.0, True).otherwise(False))

    # ── Sales tier ────────────────────────────────────────────────────────────
    # Conditions are checked in order — the first match wins.
    # Blockbuster : 10M+ sales  (huge franchise titles like Wii Sports, Mario Kart)
    # Hit         : 1M – 9.99M  (solid commercial release)
    # Mid-Tier    : 0.1M – 0.99M (smaller but viable release)
    # Long-Tail   : under 0.1M  (niche or very low-selling title)
    .withColumn('sales_tier',
        when(col('global_sales_millions') >= 10, 'Blockbuster')
        .when(col('global_sales_millions') >= 1,  'Hit')
        .when(col('global_sales_millions') >= 0.1,'Mid-Tier')
        .otherwise('Long-Tail'))
)

print('Columns after cell 6:', df_silver.columns)
display(df_silver.limit(5))


## Cell 7 — Write the Silver Delta table

We save the cleaned and enriched `df_silver` as a permanent Delta table. Key options explained:

| Option | What it does |
|---|---|
| `format('delta')` | Saves in Delta format — supports ACID, versioning, fast reads |
| `mode('overwrite')` | Replaces the table fully on each run — always a fresh clean result |
| `overwriteSchema('true')` | Allows column changes on overwrite without throwing an error |
| `partitionBy('genre')` | Splits data into sub-folders by genre — makes `WHERE genre=...` queries much faster |

`OPTIMIZE` then compacts many small files into fewer large files for better read performance.

In [0]:
# ============================================================
# CELL 7 — Write Silver Delta table partitioned by genre
# ============================================================

# .write starts the write pipeline on df_silver.
#
# .format('delta')               — save as Delta Lake format (not CSV or plain Parquet)
# .mode('overwrite')             — replace the whole table on each run
#                                  (use 'append' to add rows without deleting existing ones)
# .option('overwriteSchema','true') — allow column name/type changes on overwrite
#                                  without this, Spark throws an error if the schema changed
# .partitionBy('genre')          — split data by genre on disk
#                                  a query like WHERE genre='Sports' only reads that folder
# .saveAsTable('gaming_silver.vg_sales_clean') — saves with a queryable SQL name
df_silver.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .partitionBy('genre') \
    .saveAsTable('gaming_silver.vg_sales_clean')

# OPTIMIZE compacts many small files Delta creates during a write
# into fewer larger files — improves read speed for downstream queries.
spark.sql('OPTIMIZE gaming_silver.vg_sales_clean')
print('Silver table written and optimised.')

# Confirm total row count — should match the count printed in cell 5.
spark.sql('SELECT COUNT(*) AS silver_rows FROM gaming_silver.vg_sales_clean').show()


## Cell 8 — Validate: sales tier distribution

A final SQL query against the written Silver table to confirm everything looks correct. We group all games by their `sales_tier` and check how many titles and total sales each tier has. If any tier has zero rows or the numbers look wildly off, something went wrong in the transformation.

In [0]:
# ============================================================
# CELL 8 — Sales tier distribution validation
# ============================================================

# Run SQL directly against the saved Silver Delta table.
#
# COUNT(*) AS titles          — number of games in each tier
# SUM(global_sales_millions)  — total global sales for that tier
# ROUND(..., 2)               — round to 2 decimal places
# ORDER BY total_sales_M DESC — show highest-selling tiers first
#
# Expected approximate results:
#   Blockbuster : ~62 titles,   ~1139M total sales
#   Hit         : ~2019 titles, ~4573M total sales  (most dollars here)
#   Mid-Tier    : ~8736 titles, ~2950M total sales  (most titles here)
#   Long-Tail   : ~5781 titles, ~257M  total sales
display(spark.sql(
    'SELECT sales_tier, COUNT(*) AS titles, '
    'ROUND(SUM(global_sales_millions),2) AS total_sales_M '
    'FROM gaming_silver.vg_sales_clean '
    'GROUP BY sales_tier ORDER BY total_sales_M DESC'
))
